In [4]:
import altair as alt
import pandas as pd
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

GREY = "#676A86"

# ---------- data (from the slide) ----------
raw = pd.DataFrame(
    {
        "group": ["Overall", "18 to 34", "35 to 54", "Over 55"],
        "Yes, probably": [47, 63, 46, 33],
        "Don't know": [13, 7, 15, 17],
        "No, probably not": [40, 30, 39, 50],
    }
)
answers = ["Yes, probably", "Don't know", "No, probably not"]   # left-to-right stack order
groups = raw["group"].tolist()

long = raw.melt(id_vars="group", var_name="answer", value_name="share")
long["answer"] = pd.Categorical(long["answer"], categories=answers, ordered=True)
long = long.sort_values(["group", "answer"])

# explicit segment positions, so the labels can sit in the middle of each segment
long["end"] = long.groupby("group", sort=False)["share"].cumsum()
long["start"] = long["end"] - long["share"]
long["mid"] = (long["start"] + long["end"]) / 2

assert long.groupby("group")["share"].sum().eq(100).all()

# ---------- chart ----------
y = alt.Y("group:N", sort=groups, title=None, scale=alt.Scale(paddingInner=0.35))


color = alt.Color(
    "answer:N",
    scale=alt.Scale(domain=answers),   # domain order = palette order from the theme
    legend=alt.Legend(
        orient="bottom", columns=3, symbolLimit=0,
        columnPadding=40, title=None, labelFontSize=10,
    ),
)

bars = alt.Chart(long).mark_bar().encode(
    y=y,
    x=alt.X("start:Q", axis=None, scale=alt.Scale(domain=[0, 100])),
    x2="end:Q",
    color=color,
)
labels = alt.Chart(long).mark_text(
    align="center", baseline="middle", fontSize=12, fontWeight="bold", color="white",
).encode(y=y, x="mid:Q", text="share:Q")

caption = alt.Title(
    text="Source: Public First; Stonehaven",
    subtitle=["Share who believe government can cut taxes, cut debt and increase spending "
              "at the same time, %."],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color=GREY, subtitleColor=GREY, dy=12)

chart = (
    (bars + labels)
    .properties(width=400, height=300)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent"))

styles.save(chart, path=".", name="belief_in_magic_money", svg=True)
chart

alt.LayerChart(...)